# Klasifikasi Berita Menggunakan Model Word2Vec (Skip-Gram) dan Algoritma Gaussian Naive Bayes

### Import Library

Mengimport seluruh library yang dibutuhkan untuk pemrosesan data (pandas, numpy), scraping (requests, BeautifulSoup), pembuatan vektor teks (Word2Vec), pembuatan model machine learning (GaussianNB), evaluasi, serta pengelolaan file.

In [ ]:
!pip install -q gensim beautifulsoup4 requests joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 39.7 MB/s eta 0:00:00


In [ ]:
import os
import re
import joblib
import requests
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Memuat Dataset

Membaca dataset artikel berita dari file Excel (berita_detik_200_artikel.xlsx), menampilkan total jumlah baris data, dan melihat 5 sampel data teratas.

In [ ]:
file_path = "berita_detik_200_artikel.xlsx"

df = pd.read_excel(file_path)
print("Jumlah data:", len(df))
df.head()

Jumlah data: 200


,id,isi_berita,label
0,1,Kejuaraan Nasional Gokart 2026 yang berlangsun...,sport
1,2,PT Bank Negara Indonesia (Persero) Tbk atau BN...,sport
2,3,"Pelatih Timnas voli putra Indonesia, Reidel To...",sport
3,4,Kontingen Indonesia bersiap menatap Asian Game...,sport
4,5,Rizki Juniansyah siap menjalani debut di Asian...,sport


### Pemeriksaan Struktur Data

Menampilkan daftar nama kolom yang ada di dataset serta jumlah distribusi data untuk setiap kategori/label berita.

In [ ]:
print("Kolom:")
print(df.columns.tolist())

print("\nJumlah data setiap label:")
print(df["label"].value_counts())

Kolom:
['id', 'isi_berita', 'label']

Jumlah data setiap label:
label
sport      100
finance    100
Name: count, dtype: int64


### Membuat Fungsi Preprocessing Teks

Mendefinisikan fungsi preprocessing() untuk membersihkan teks mentah, meliputi pembersihan huruf kecil (case folding), penghapusan URL, angka, tanda baca, serta spasi berlebih.

In [ ]:
def preprocessing(teks):
    teks = str(teks).lower()

    # Menghapus URL
    teks = re.sub(r"http\S+|www\S+", "", teks)

    # Menghapus angka
    teks = re.sub(r"\d+", "", teks)

    # Menghapus tanda baca
    teks = re.sub(r"[^a-z\s]", " ", teks)

    # Menghapus spasi berlebih
    teks = re.sub(r"\s+", " ", teks).strip()

    return teks

### Penerapan Preprocessing dan Tokenisasi

Menerapkan fungsi pembersihan ke seluruh isi berita (menjadi kolom teks_bersih) lalu memecah teks menjadi daftar kata/token (kolom tokens).

In [ ]:
df["teks_bersih"] = df["isi_berita"].apply(preprocessing)

df["tokens"] = df["teks_bersih"].apply(lambda x: str(x).split())

df[["isi_berita", "teks_bersih", "label"]].head()

,isi_berita,teks_bersih,label
0,Kejuaraan Nasional Gokart 2026 yang berlangsun...,kejuaraan nasional gokart yang berlangsung dal...,sport
1,PT Bank Negara Indonesia (Persero) Tbk atau BN...,pt bank negara indonesia persero tbk atau bni ...,sport
2,"Pelatih Timnas voli putra Indonesia, Reidel To...",pelatih timnas voli putra indonesia reidel toi...,sport
3,Kontingen Indonesia bersiap menatap Asian Game...,kontingen indonesia bersiap menatap asian game...,sport
4,Rizki Juniansyah siap menjalani debut di Asian...,rizki juniansyah siap menjalani debut di asian...,sport


### Pembagian Data Train dan Test

Membagi dataset menjadi data latih (training 80%) dan data uji (testing 20%) menggunakan metode stratified sampling agar proporsi label tetap seimbang.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["tokens"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Data training :", len(X_train))
print("Data testing  :", len(X_test))

Data training : 160
Data testing  : 40


### Pelatihan Model Word2Vec (Skip-gram)

Melatih model Word2Vec menggunakan arsitektur Skip-gram (sg=1) pada token data training untuk mempelajari representasi vektor dari setiap kata.

In [ ]:
model_w2v = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=42,
    epochs=20
)

print("Skip-gram berhasil dibuat dengan kosakata lengkap.")

Skip-gram berhasil dibuat dengan kosakata lengkap.


### Membuat Fungsi Vektorisasi Dokumen

Mendefinisikan fungsi document_vector() untuk merata-ratakan (mean pooling) seluruh vektor kata dalam satu dokumen/berita menjadi satu vektor representasi tunggal.

In [ ]:
def document_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

### Konversi Teks ke Vektor Dokumen

Mengubah kumpulan token pada X_train dan X_test menjadi bentuk matriks/array fitur bernilai numerik agar dapat diproses oleh algoritma klasifikasi.

In [ ]:
X_train_vector = np.array([
    document_vector(tokens, model_w2v)
    for tokens in X_train
])

X_test_vector = np.array([
    document_vector(tokens, model_w2v)
    for tokens in X_test
])

print("Ukuran data training:", X_train_vector.shape)
print("Ukuran data testing :", X_test_vector.shape)

Ukuran data training: (160, 100)
Ukuran data testing : (40, 100)


### Pelatihan Model Naive Bayes

Melatih algoritma Gaussian Naive Bayes (model_nb) menggunakan matriks vektor fitur data training dan labelnya.

In [ ]:
model_nb = GaussianNB()

model_nb.fit(
    X_train_vector,
    y_train
)

print("Naive Bayes berhasil dilatih.")

Naive Bayes berhasil dilatih.


### Prediksi dan Pengukuran Akurasi

Melakukan prediksi pada data testing menggunakan model Naive Bayes yang telah dilatih, lalu menghitung persentase akurasi model.

In [ ]:
y_pred = model_nb.predict(X_test_vector)

akurasi = accuracy_score(y_test, y_pred)

print("Akurasi:", akurasi)

Akurasi: 0.975


### Evaluasi Classification Report

Menampilkan laporan evaluasi detail meliputi nilai Precision, Recall, dan F1-Score untuk setiap kelas/kategori berita.

In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     finance       0.95      1.00      0.98        20
       sport       1.00      0.95      0.97        20

    accuracy                           0.97        40
   macro avg       0.98      0.97      0.97        40
weighted avg       0.98      0.97      0.97        40



### Evaluasi Confusion Matrix

Menampilkan matriks konfusi untuk melihat detail jumlah prediksi yang benar dan prediksi yang salah (misclassification) pada data uji.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[20  0]
 [ 1 19]]


### Pengujian Model dengan Teks Manual

Menguji kinerja model secara langsung menggunakan contoh teks berita buatan sendiri untuk melihat hasil klasifikasi kategorinya.

In [ ]:
berita_baru = """
Tim sepak bola berhasil memenangkan pertandingan setelah mencetak
dua gol pada babak kedua.
"""

teks_baru = preprocessing(berita_baru)
tokens_baru = teks_baru.split()

vektor_baru = document_vector(tokens_baru, model_w2v)

hasil = model_nb.predict([vektor_baru])

print("Hasil klasifikasi:", hasil[0])

Hasil klasifikasi: sport


### Membuat Fungsi Scraping Berita

Mendefinisikan fungsi ambil_berita_dari_url() menggunakan requests dan BeautifulSoup untuk mengambil teks paragraf utama dari tautan berita detik.com.

In [ ]:
def ambil_berita_dari_url(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Cari kontainer utama berita (Detik, Kompas, dll)
        isi = (
            soup.select_one("div.detail__body-text") or
            soup.select_one("div.read__content") or
            soup.select_one("div.detail__body") or
            soup.select_one("article")
        )

        if isi:
            paragraphs = isi.find_all("p")
            teks = " ".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
            return teks if teks else isi.get_text(separator=" ", strip=True)

        # Fallback universal jika selector spesifik tidak ditemukan
        paragraphs = soup.find_all("p")
        teks_paragraf = " ".join([p.get_text(strip=True) for p in paragraphs])
        return teks_paragraf if len(teks_paragraf) > 100 else None

    except Exception as e:
        return None

### Pengujian Scraping URL

Menerima masukan URL dari pengguna, menjalankan fungsi scraping, dan menampilkan 1.000 karakter pertama dari teks berita yang berhasil diambil.

In [ ]:
url = input("Masukkan URL berita Detik: ")

berita = ambil_berita_dari_url(url)

if berita:
    print(berita[:1000])
else:
    print("Isi berita tidak ditemukan.")

Masukkan URL berita Detik: https://sport.detik.com/moto-gp/d-8693221/jadwal-motogp-mandalika-2026-misi-marc-marquez-kejar-jorge-martin
Pertamina Grand Prix of Indonesia 2026 di depan mata. Seri ke-17 MotoGP 2026 ini berlangsung di Pertamina Mandalika International Circuit, Lombok, Nusa Tenggara Barat, pada 9-11 Oktober 2026. MotoGP Mandalika menjadi seri penting dalam persaingan gelar juara dunia. Jorge Martin masih memimpin klasemen dengan 333 poin, tetapi Marc Marquez terus membuntutinya dengan 331 poin. Keduanya hanya terpaut dua angka setelah Marc menyapu bersih Sprint Race dan balapan utama MotoGP Jepang di Motegi.Marc meraih 37 poin dari akhir pekan MotoGP Jepang. SCROLL TO CONTINUE WITH CONTENT Hasil tersebut memangkas jarak dengan Martin secara signifikan dan membuat persaingan menuju gelar semakin ketat. Sementara itu, Marco Bezzecchi berada di posisi ketiga dengan 284 poin atau tertinggal 49 poin dari Martin. Kini, persaingan berlanjut ke Mandalika. Marc akan berusaha melanju

### Membuat Pipeline Klasifikasi Berita dari URL

Mendefinisikan fungsi klasifikasi_url() yang menggabungkan seluruh alur secara otomatis: scraping URL $\rightarrow$ preprocessing $\rightarrow$ vektorisasi Word2Vec $\rightarrow$ prediksi Naive Bayes.

In [ ]:
def klasifikasi_url(url):

    berita = ambil_berita_dari_url(url)

    if berita is None:
        return "Isi berita tidak ditemukan."

    teks = preprocessing(berita)
    tokens = teks.split()

    vektor = document_vector(
        tokens,
        model_w2v
    )

    hasil = model_nb.predict([vektor])[0]

    return hasil

### Pengujian Klasifikasi Berita Langsung dari URL

Meminta masukan URL berita Detik dari pengguna dan mencetak hasil akhir kategori beritanya secara otomatis.

In [ ]:
url = input("Masukkan URL berita Detik: ")

hasil = klasifikasi_url(url)

print("Kategori berita:", hasil)

Masukkan URL berita Detik: https://sport.detik.com/moto-gp/d-8693221/jadwal-motogp-mandalika-2026-misi-marc-marquez-kejar-jorge-martin
Kategori berita: sport


### Menyimpan dan Mengunduh Model

In [ ]:
os.makedirs("model", exist_ok=True)

joblib.dump(
    model_w2v,
    "model/w2v_model.pkl"
)

joblib.dump(
    model_nb,
    "model/nb_model.pkl"
)

print("Model berhasil disimpan.")

Model berhasil disimpan.


In [ ]:
from google.colab import files

files.download("model/w2v_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("model/nb_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>